[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/22_anomaly_detection/01_statistical_and_gaussian_methods.ipynb)

# Bài 01 — Phát hiện bất thường: phương pháp thống kê và mô hình Gaussian

**🎯 Mục tiêu**
- Các loại bất thường: điểm (point), ngữ cảnh (contextual), tập thể (collective); khi nào dùng anomaly detection thay vì phân loại.
- Phương pháp đơn biến: z-score, **robust z-score (MAD)**, IQR — và vì sao outlier "che" chính nó.
- **Mô hình Gaussian** (khoá ML của Andrew Ng): ước lượng $p(\mathbf x) = \prod_j \mathcal N(x_j; \mu_j, \sigma_j^2)$ trên dữ liệu bình thường; cảnh báo khi $p(\mathbf x) < \varepsilon$; **chọn ε bằng F1** trên validation có nhãn.
- **Gaussian nhiều chiều** và khoảng cách Mahalanobis — bắt được bất thường do **tương quan**; ước lượng robust.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/22_anomaly_detection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import multivariate_normal
from sklearn.covariance import EllipticEnvelope, MinCovDet
from sklearn.metrics import f1_score, precision_score, recall_score

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Bất thường là gì? Khi nào dùng anomaly detection?

| Loại | Ví dụ |
|---|---|
| **Điểm** | giao dịch 500 triệu từ thẻ thường chỉ chi vài trăm nghìn |
| **Ngữ cảnh** | 35°C bình thường vào tháng 6, bất thường vào tháng 12 |
| **Tập thể** | chuỗi đăng nhập thất bại liên tục — từng lần thì bình thường |

**Anomaly detection** (học mô hình của dữ liệu **bình thường**) phù hợp khi: positive **cực hiếm**, **ít hoặc không có nhãn**, và các kiểu bất thường **đa dạng/thay đổi** (kiểu tấn công mới chưa từng thấy). Khi có nhiều nhãn và bất thường lặp lại kiểu cũ → **phân loại có giám sát** thường tốt hơn (chương 19).

## 2. Đơn biến: z-score, robust z-score, IQR

In [ ]:
latency = np.r_[rng.normal(100, 10, 990), rng.normal(300, 30, 10)]            # ms; 10 request bất thường
z = (latency - latency.mean()) / latency.std()
med = np.median(latency)
mad = np.median(np.abs(latency - med))
robust_z = 0.6745 * (latency - med) / mad                                       # 0.6745: để MAD ≈ σ với dữ liệu chuẩn
q1, q3 = np.percentile(latency, [25, 75])
iqr_flag = (latency < q1 - 1.5 * (q3 - q1)) | (latency > q3 + 1.5 * (q3 - q1))
truth = np.r_[np.zeros(990, bool), np.ones(10, bool)]
for name, flag in [("|z| > 3", np.abs(z) > 3), ("|robust z| > 3.5", np.abs(robust_z) > 3.5), ("IQR (1.5×)", iqr_flag)]:
    print(f"{name:17s} cảnh báo {flag.sum():3d} | bắt {flag[truth].sum()}/10 bất thường | báo nhầm {flag[~truth].sum()}")

Với **nhiều** outlier hơn, chúng kéo trung bình và **thổi phồng độ lệch chuẩn** → z-score của chính chúng nhỏ lại ("masking"). Median và MAD gần như không bị ảnh hưởng:

In [ ]:
latency2 = np.r_[rng.normal(100, 10, 900), rng.normal(300, 30, 100)]           # 10% bất thường
z2 = (latency2 - latency2.mean()) / latency2.std()
med2, mad2 = np.median(latency2), np.median(np.abs(latency2 - np.median(latency2)))
print(f"10% bất thường — std = {latency2.std():.1f} (thật ≈ 10) | bắt bằng |z|>3: {(np.abs(z2[900:]) > 3).sum()}/100 | "
      f"bằng |robust z|>3.5: {(np.abs(0.6745 * (latency2[900:] - med2) / mad2) > 3.5).sum()}/100")

## 3. Mô hình Gaussian và chọn ε bằng F1

Ví dụ kinh điển (khoá Machine Learning của Andrew Ng): giám sát **máy chủ** qua hai feature — thông lượng (MB/s) và độ trễ (ms).

**Quy trình:**
1. **Train**: chỉ dữ liệu (gần như) bình thường, **không cần nhãn** → ước lượng $\mu_j, \sigma_j^2$ cho từng feature.
2. $p(\mathbf x) = \prod_j \frac{1}{\sqrt{2\pi}\sigma_j}\exp\left(-\frac{(x_j - \mu_j)^2}{2\sigma_j^2}\right)$.
3. **Validation có nhãn** (vài bất thường đã biết): chọn ngưỡng $\varepsilon$ tối đa **F1** — accuracy vô nghĩa khi bất thường hiếm.
4. **Test**: báo bất thường khi $p(\mathbf x) < \varepsilon$.

In [ ]:
def make_servers(n_normal, n_anom, seed):
    r = np.random.default_rng(seed)
    normal = r.multivariate_normal([14, 15], [[1.6, 0.4], [0.4, 1.4]], n_normal)
    anomalies = np.c_[r.uniform(5, 25, 3 * n_anom), r.uniform(5, 25, 3 * n_anom)]
    anomalies = anomalies[np.linalg.norm(anomalies - [14, 15], axis=1) > 2.5][:n_anom]   # lệch khỏi vùng bình thường
    X = np.vstack([normal, anomalies])
    return X, np.r_[np.zeros(len(normal), int), np.ones(len(anomalies), int)]


X_train, _ = make_servers(1000, 0, seed=1)                      # chỉ máy bình thường, không nhãn
X_val, y_val = make_servers(300, 30, seed=2)                    # vài chục sự cố đã biết — đủ để chọn ε
X_test, y_test = make_servers(300, 30, seed=3)


def fit_gaussian(X):
    return X.mean(0), X.var(0)


def gaussian_density(X, mu, var):
    return np.prod(np.exp(-((X - mu) ** 2) / (2 * var)) / np.sqrt(2 * np.pi * var), axis=1)


def select_threshold(y_true, p):
    best_eps, best_f1 = 0.0, 0.0
    for eps in np.linspace(p.min(), p.max(), 1000):
        f1 = f1_score(y_true, p < eps, zero_division=0)
        if f1 > best_f1:
            best_eps, best_f1 = eps, f1
    return best_eps, best_f1


mu, var = fit_gaussian(X_train)
eps, f1_val = select_threshold(y_val, gaussian_density(X_val, mu, var))
pred = gaussian_density(X_test, mu, var) < eps
print(f"μ = {mu.round(2)}, σ² = {var.round(2)} | ε = {eps:.2e} (F1 validation = {f1_val:.3f})")
print(f"test: precision = {precision_score(y_test, pred):.3f} | recall = {recall_score(y_test, pred):.3f} | F1 = {f1_score(y_test, pred):.3f}")

In [ ]:
gx, gy = np.meshgrid(np.linspace(4, 26, 300), np.linspace(4, 26, 300))
P = gaussian_density(np.c_[gx.ravel(), gy.ravel()], mu, var).reshape(gx.shape)
fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(gx, gy, P, levels=[eps], colors="r", linewidths=2)
ax.contourf(gx, gy, np.log(P + 1e-300), levels=30, cmap="Blues", alpha=0.5)
ax.scatter(*X_test[y_test == 0].T, s=8, label="bình thường")
ax.scatter(*X_test[y_test == 1].T, s=30, marker="x", color="k", label="bất thường thật")
ax.scatter(*X_test[pred].T, s=120, facecolors="none", edgecolors="r", label="bị cảnh báo")
ax.set_xlabel("thông lượng (MB/s)")
ax.set_ylabel("độ trễ (ms)")
ax.legend(fontsize=8)
ax.set_title("Đường đỏ: p(x) = ε")
plt.show()

> 📝 Mẹo từ khoá học: feature của mô hình Gaussian nên có phân phối **gần chuẩn** — biến đổi feature lệch ($\log x$, $\sqrt x$, $x^{1/3}$); khi một kiểu bất thường bị bỏ sót, tìm **feature mới** làm nó nổi bật (vd tỉ số CPU / lưu lượng mạng). Với nhiều feature, $p(\mathbf x)$ trải dài nhiều bậc độ lớn — duyệt ε trên **thang log** (hoặc làm việc với $\log p$) thay vì lưới đều trên $p$ (bài tập 9).

## 4. Gaussian nhiều chiều: bắt bất thường do tương quan

Mô hình tích các Gaussian đơn biến giả định feature **độc lập** → đường đồng mức là ellipse **song song trục**. Nếu hai feature tương quan mạnh, một điểm có từng giá trị bình thường nhưng **kết hợp** bất thường (CPU cao, lưu lượng thấp) sẽ bị bỏ sót.

**Gaussian nhiều chiều**: $p(\mathbf x) = \frac{1}{(2\pi)^{d/2}|\Sigma|^{1/2}}\exp\left(-\frac12(\mathbf x - \boldsymbol\mu)^\top\Sigma^{-1}(\mathbf x - \boldsymbol\mu)\right)$ — số mũ là bình phương **khoảng cách Mahalanobis**.

In [ ]:
cov = [[1, 0.9], [0.9, 1]]
Xn = rng.multivariate_normal([0, 0], cov, 1000)                   # CPU và lưu lượng tương quan 0.9
odd = np.array([[1.5, -1.5], [-1.4, 1.6], [1.8, -1.2]])          # từng giá trị bình thường, kết hợp lạ
X_all = np.vstack([Xn, odd])
mu_i, var_i = fit_gaussian(Xn)
p_indep = gaussian_density(X_all, mu_i, var_i)
p_multi = multivariate_normal(Xn.mean(0), np.cov(Xn.T)).pdf(X_all)
for name, p in [("Gaussian độc lập", p_indep), ("Gaussian nhiều chiều", p_multi)]:
    rank = (p[:-3, None] < p[-3:][None]).sum(0) + 1               # 1 + số điểm bình thường có mật độ thấp hơn
    print(f"{name:21s} thứ hạng bất thường của 3 điểm lạ (1 = bất thường nhất, trên {len(p)} điểm): {rank.tolist()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
g = np.linspace(-4, 4, 200)
GX, GY = np.meshgrid(g, g)
grid = np.c_[GX.ravel(), GY.ravel()]
for ax, (name, Pg) in zip(axes, [("độc lập (ellipse song song trục)", gaussian_density(grid, mu_i, var_i)),
                                 ("nhiều chiều (ellipse theo tương quan)", multivariate_normal(Xn.mean(0), np.cov(Xn.T)).pdf(grid))]):
    ax.contour(GX, GY, Pg.reshape(GX.shape), levels=6, cmap="viridis")
    ax.scatter(*Xn.T, s=4, alpha=0.3)
    ax.scatter(*odd.T, s=80, marker="x", color="r", label="kết hợp bất thường")
    ax.set_title(f"Gaussian {name}")
    ax.set_aspect("equal")
axes[0].legend()
plt.show()

| | Gaussian độc lập | Gaussian nhiều chiều |
|---|---|---|
| Tương quan | không bắt được (cần tự tạo feature như tỉ số) | tự động |
| Tham số | $2d$ | $d + d(d+1)/2$ |
| Yêu cầu | — | $n > d$ (nên $n \ge 10d$), $\Sigma$ khả nghịch |
| Chi phí | rẻ, $d$ lớn vẫn ổn | nghịch đảo $\Sigma$: $O(d^3)$ |

## 5. Ước lượng robust

Nếu dữ liệu train có lẫn bất thường, $\mu$ và $\Sigma$ bị kéo lệch. **Minimum Covariance Determinant** (MCD) ước lượng trên tập con "sạch" nhất; `EllipticEnvelope` dùng MCD để phát hiện bất thường:

In [ ]:
contaminated = np.vstack([Xn, rng.multivariate_normal([3, -3], [[0.3, 0], [0, 0.3]], 80)])     # 7% bất thường lẫn vào train
emp_cov = np.cov(contaminated.T)
mcd = MinCovDet(random_state=0).fit(contaminated)
print(f"tương quan thật = 0.90 | ước lượng thường = {emp_cov[0, 1] / np.sqrt(emp_cov[0, 0] * emp_cov[1, 1]):.2f} | "
      f"MCD = {mcd.covariance_[0, 1] / np.sqrt(mcd.covariance_[0, 0] * mcd.covariance_[1, 1]):.2f}")
ee = EllipticEnvelope(contamination=0.07, random_state=0).fit(contaminated)
print(f"EllipticEnvelope gắn cờ {(ee.predict(contaminated[-80:]) == -1).mean():.0%} điểm bất thường lẫn trong train")

## 🧠 Kiểm tra nhanh

1. Khi nào dùng anomaly detection thay vì phân loại có giám sát?
2. Vì sao z-score kém khi có nhiều outlier? Robust z-score khắc phục thế nào?
3. Vì sao chọn ε bằng F1 thay vì accuracy?
4. Một điểm có CPU cao và lưu lượng mạng thấp — từng giá trị đều bình thường. Mô hình nào phát hiện được?

## 📌 Tóm tắt

- Anomaly detection = mô hình hoá dữ liệu bình thường; hợp khi bất thường hiếm, ít nhãn, đa dạng.
- Đơn biến: robust z-score (median/MAD), IQR — tránh masking.
- Gaussian: $p(\mathbf x) < \varepsilon$, ε chọn bằng F1 trên validation; Gaussian nhiều chiều (Mahalanobis) bắt bất thường do tương quan; MCD cho ước lượng robust.

## 📚 Tài liệu tham khảo

- Andrew Ng — Machine Learning (Coursera), Week 9: Anomaly Detection
- Chandola, Banerjee & Kumar (2009) — *Anomaly Detection: A Survey*
- Rousseeuw & Van Driessen (1999) — *A Fast Algorithm for the Minimum Covariance Determinant Estimator*